# 02 - Force Field and Interaction Maps

This notebook introduces the MPIPI-GG style residue-residue interaction model and generates FINCHES-style interaction maps for FUS LCD variants.

## Contents
1. Explore the force field parameters
2. Visualize the 20x20 interaction matrix
3. Generate raw interaction maps for all variants
4. Apply Gaussian smoothing
5. Normalize and compare intermaps

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import modules
from src.sequences import VARIANTS, get_variant
from src.forcefield import (
    get_default_matrix,
    get_interaction_energy,
    get_most_attractive_pairs,
    compute_stickiness_ranking,
    print_matrix_summary,
    ForceFieldParameters,
    build_interaction_matrix,
    AA_ORDER,
)
from src.intermaps import (
    compute_interaction_map,
    smooth_interaction_map,
    normalize_interaction_map,
    generate_finches_intermap,
    compute_all_intermaps,
    compute_map_statistics,
    InterMapConfig,
    save_intermaps,
)
from src.plotting import (
    plot_interaction_matrix,
    plot_interaction_map,
    plot_interaction_maps_comparison,
    set_publication_style,
)

In [ ]:
set_publication_style()

## 1. Force Field Parameters

The MPIPI-GG style force field captures key interactions driving IDP phase separation:
- **Aromatic-aromatic (pi-pi)**: Y-Y, F-F, W-W stacking
- **Cation-pi**: R/K with aromatic residues
- **Electrostatic**: Charge-charge interactions
- **Hydrophobic**: Non-polar contacts

All energies are in units of kT (thermal energy at ~300K).

In [ ]:
# Display force field parameters
params = ForceFieldParameters()

print("MPIPI-GG Style Force Field Parameters:")
print("=" * 50)
print(f"Aromatic-aromatic (pi-pi): {params.epsilon_aromatic:.3f} kT")
print(f"Cation-pi:                 {params.epsilon_cation_pi:.3f} kT")
print(f"Electrostatic attractive:  {params.epsilon_attractive:.3f} kT")
print(f"Electrostatic repulsive:   {params.epsilon_repulsive:.3f} kT")
print(f"Hydrophobic:               {params.epsilon_hydrophobic:.3f} kT")
print(f"Polar:                     {params.epsilon_polar:.3f} kT")
print(f"Glycine factor:            {params.glycine_factor:.3f}")
print(f"Proline penalty:           {params.proline_penalty:.3f} kT")

In [ ]:
# Key interactions for FUS LCD
print("\nKey Interaction Energies:")
print("=" * 40)

pairs = [
    ("Y", "Y", "Tyr-Tyr (aromatic)"),
    ("Y", "F", "Tyr-Phe (aromatic)"),
    ("Y", "R", "Tyr-Arg (cation-pi)"),
    ("Y", "K", "Tyr-Lys (cation-pi)"),
    ("Y", "S", "Tyr-Ser"),
    ("Y", "G", "Tyr-Gly (spacer)"),
    ("R", "D", "Arg-Asp (salt bridge)"),
    ("G", "G", "Gly-Gly (flexible)"),
    ("S", "S", "Ser-Ser (polar)"),
]

for aa1, aa2, desc in pairs:
    energy = get_interaction_energy(aa1, aa2)
    print(f"{desc:25s}: {energy:+.3f} kT")

## 2. The 20x20 Interaction Matrix

In [ ]:
# Get and display the interaction matrix
matrix = get_default_matrix()
print_matrix_summary(matrix)

In [ ]:
# Visualize the full matrix
fig = plot_interaction_matrix(
    matrix,
    title="MPIPI-GG Style Residue-Residue Interaction Matrix",
    figsize=(10, 8)
)
plt.show()

In [ ]:
# Residue stickiness ranking
stickiness = compute_stickiness_ranking(matrix)

fig, ax = plt.subplots(figsize=(12, 5))
names = list(stickiness.keys())
values = list(stickiness.values())

# Color by residue type
colors = []
for aa in names:
    if aa in 'YFW':
        colors.append('#E74C3C')  # Aromatic
    elif aa in 'RK':
        colors.append('#3498DB')  # Cationic
    elif aa in 'DE':
        colors.append('#2ECC71')  # Anionic
    else:
        colors.append('#95A5A6')  # Other

ax.bar(names, values, color=colors, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Amino Acid')
ax.set_ylabel('Mean Interaction Energy (kT)')
ax.set_title('Residue Stickiness Ranking\n(More negative = stickier)')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E74C3C', label='Aromatic (Y,F,W)'),
    Patch(facecolor='#3498DB', label='Cationic (R,K)'),
    Patch(facecolor='#2ECC71', label='Anionic (D,E)'),
    Patch(facecolor='#95A5A6', label='Other'),
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 3. Generate Interaction Maps

For each variant, we compute an NxN interaction map where I[i,j] is the interaction energy between residue i and residue j based on their amino acid types.

In [ ]:
# Compute raw interaction maps for all variants
raw_intermaps = {}

for name, variant in VARIANTS.items():
    raw_intermaps[name] = compute_interaction_map(variant)
    print(f"{name}: shape = {raw_intermaps[name].shape}")

In [ ]:
# Statistics for WT raw map
wt_stats = compute_map_statistics(raw_intermaps['WT'])

print("WT Raw Interaction Map Statistics:")
print("=" * 40)
for key, value in wt_stats.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")

In [ ]:
# Visualize WT raw interaction map
fig = plot_interaction_map(
    raw_intermaps['WT'],
    title="FUS LCD Wild-Type: Raw Interaction Map",
    colorbar_label="Interaction Energy (kT)"
)
plt.show()

## 4. Gaussian Smoothing

Smoothing captures local interaction neighborhoods and reduces noise from individual residue pairs. The smoothing width (sigma) controls the spatial scale of interactions.

In [ ]:
# Compare different smoothing widths
sigmas = [0, 1, 2, 3]
wt_raw = raw_intermaps['WT']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, sigma in zip(axes.flat, sigmas):
    if sigma == 0:
        smoothed = wt_raw
        title = "Raw (no smoothing)"
    else:
        smoothed = smooth_interaction_map(wt_raw, sigma=sigma)
        title = f"Sigma = {sigma}"
    
    abs_max = max(abs(smoothed.min()), abs(smoothed.max()))
    im = ax.imshow(smoothed, cmap='RdBu_r', vmin=-abs_max, vmax=abs_max)
    ax.set_title(title)
    ax.set_xlabel('Residue')
    ax.set_ylabel('Residue')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Effect of Gaussian Smoothing on FUS LCD Interaction Map', fontsize=14)
plt.tight_layout()
plt.show()

## 5. FINCHES-Style Processing Pipeline

The full pipeline: Raw → Smoothed → Normalized

In [ ]:
# Configure the processing pipeline
config = InterMapConfig(
    smooth=True,
    sigma=2.0,
    normalize=True,
    normalize_method='zscore',
    mask_diagonal=False,
)

print("FINCHES Intermap Configuration:")
print(f"  Smoothing: {config.smooth} (sigma={config.sigma})")
print(f"  Normalization: {config.normalize} ({config.normalize_method})")

In [ ]:
# Compute FINCHES-style intermaps for all variants
intermaps = compute_all_intermaps(VARIANTS, config=config)

print("Processed Interaction Maps:")
print("=" * 50)
for name, imap in intermaps.items():
    stats = compute_map_statistics(imap)
    print(f"\n{name}:")
    print(f"  Mean: {stats['mean']:.3f}, Std: {stats['std']:.3f}")
    print(f"  Range: [{stats['min']:.3f}, {stats['max']:.3f}]")

In [ ]:
# Compare WT vs AllY_to_S
fig = plot_interaction_maps_comparison(
    {"WT": intermaps["WT"], "AllY_to_S": intermaps["AllY_to_S"]},
    ncols=2,
    figsize=(12, 5),
    shared_scale=True
)
plt.suptitle('FINCHES Intermaps: WT vs All-Tyrosine-to-Serine', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Full comparison panel
fig = plot_interaction_maps_comparison(
    intermaps,
    ncols=3,
    figsize=(15, 10),
    shared_scale=True
)
plt.suptitle('FINCHES-Style Interaction Maps for All FUS LCD Variants', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Save intermaps for downstream analysis
output_path = Path("../data/outputs/intermaps.npz")
output_path.parent.mkdir(parents=True, exist_ok=True)
save_intermaps(intermaps, str(output_path))
print(f"Intermaps saved to {output_path}")

## Summary

In this notebook we:
1. Explored the MPIPI-GG style force field (aromatic, cation-pi, electrostatic interactions)
2. Visualized the 20x20 amino acid interaction matrix
3. Generated raw interaction maps for all FUS LCD variants
4. Applied Gaussian smoothing to capture local neighborhoods
5. Computed normalized FINCHES-style intermaps

**Key observation**: The AllY_to_S variant shows dramatically reduced attractive interactions compared to WT, consistent with the loss of tyrosine-mediated sticker contacts.

**Next**: Notebook 03 - Sliding Window and Sticker Masks